# 05 · Self-Attention

**Belgedeki bölüm:** Self Attention

- Her token, cümledeki diğer token'larla ne kadar alakalı olduğunu hesaplar.
- Benzerlik: vektörlerin nokta çarpımı (dot product) → `Q·Kᵀ` matrisi.
- Softmax ile her satır 0–1 arasına çekilir ve toplamı 1 olur (attention weights).
- Ağırlıklar değerlerle çarpılır → context vector: kelimenin cümleye göre zenginleşmiş anlamı.
- Bu notebook önce Wq, Wk, Wv matrisleri olmadan doğrudan embedding'lerle hesaplar; Q, K, V 06'da gelir.

In [ ]:
import torch
from usta_embedding import UstaEmbedding
from usta_tokenizer import UstaTokenizer

u_tokenizer = UstaTokenizer("tokenizer.json")

prompt = "the capital of united states and the capital of france"

tokens = u_tokenizer.encode(prompt)

# UstaEmbedding = nn.Embedding + RoPE pozisyon bilgisi (04_Positional_Encoding, usta_embedding.py)
torch.manual_seed(1)
u_embedding = UstaEmbedding(vocab_size=len(u_tokenizer.vocab), embedding_dim=4, context_length=32)

sentence_meanings = u_embedding(tokens)
sentence_meanings.shape

In [ ]:
u_tokenizer.tokenize(prompt)

In [ ]:
from plot_tokens import plot_tokens

u_sentences = [
  {
    "words": sentence_meanings.detach().numpy(),
    "labels": u_tokenizer.tokenize(prompt),
    "color": "red",
  },
]

plot_tokens(u_sentences, "Models Context Space")

In [ ]:
sentence_meanings

In [ ]:
the_pos = [-1.5256, -0.7502, -0.6540, -1.6095]
capital_pos = [ 0.9326, -0.2774,  0.3166, -1.6980]

Manhattan Mesafesi ile Self-Attention Hesaplama

In [ ]:
hardness_dist = abs(the_pos[0] - capital_pos[0])
brightness_dist = abs(the_pos[1] - capital_pos[1])
redness_dist = abs(the_pos[2] - capital_pos[2])
blueness_dist = abs(the_pos[3] - capital_pos[3])

hardness_dist, brightness_dist, redness_dist, blueness_dist

In [ ]:
total_dist = hardness_dist + brightness_dist + redness_dist + blueness_dist
total_dist

In [ ]:
apple = [-1.5256, -0.7502, -0.6540, -1.6095]
real_apple = [0.5, -0.7502, -0.6540, -1.6095]

In [ ]:
def is_apple(pos, real_pos):
  dist1 = pos[0] - real_pos[0]

  print(dist1)

  if dist1 > 0:
    apple[0] -= 0.5
  else:
    apple[0] += 0.5


  return dist1 > 0 and dist1 < 0.5

is_apple(apple, real_apple)

Dot Product ile Self-Attention Hesaplama

Kosinüs benzerliği olması için bu dot product sonucunu, vektörlerin büyüklüklerine (normlarına) bölmen gerekir

In [ ]:
the_pos = [-1.5256, -0.7502, -0.6540, -1.6095]
capital_pos = [0.9326, -0.2774,  0.3166, -1.6980]

O tokenlara ait listelerdeki indikslerle çarpma yap. the_pos[0]*capital_pos[0],the_pos[1]*capital_pos[1] gibi

In [ ]:
cos_sim_hardness = the_pos[0] * capital_pos[0]
cos_sim_brightness = the_pos[1] * capital_pos[1]
cos_sim_redness = the_pos[2] * capital_pos[2]
cos_sim_blueness = the_pos[3] * capital_pos[3]

total_cos_sim = cos_sim_hardness + cos_sim_brightness + cos_sim_redness + cos_sim_blueness

cos_sim_hardness, cos_sim_brightness, cos_sim_redness, cos_sim_blueness, total_cos_sim

In [ ]:
#the-the benzerliği
cs_0_0 = sentence_meanings[0][0] * sentence_meanings[0][0] + sentence_meanings[0][1] * sentence_meanings[0][1] + sentence_meanings[0][2] * sentence_meanings[0][2] + sentence_meanings[0][3] * sentence_meanings[0][3]
#the-boşlukla benzerliği
cs_0_1 = sentence_meanings[0][0] * sentence_meanings[1][0] + sentence_meanings[0][1] * sentence_meanings[1][1] + sentence_meanings[0][2] * sentence_meanings[1][2] + sentence_meanings[0][3] * sentence_meanings[1][3]
cs_0_2 = sentence_meanings[0][0] * sentence_meanings[2][0] + sentence_meanings[0][1] * sentence_meanings[2][1] + sentence_meanings[0][2] * sentence_meanings[2][2] + sentence_meanings[0][3] * sentence_meanings[2][3]
cs_0_3 = sentence_meanings[0][0] * sentence_meanings[3][0] + sentence_meanings[0][1] * sentence_meanings[3][1] + sentence_meanings[0][2] * sentence_meanings[3][2] + sentence_meanings[0][3] * sentence_meanings[3][3]

cs_0_0, cs_0_1, cs_0_2, cs_0_3

In [ ]:
the_similarities = []

for i in range(sentence_meanings.shape[0]):
  cs_the_i = sentence_meanings[0][0] * sentence_meanings[i][0] + sentence_meanings[0][1] * sentence_meanings[i][1] + sentence_meanings[0][2] * sentence_meanings[i][2] + sentence_meanings[0][3] * sentence_meanings[i][3]
  the_similarities.append(cs_the_i)

the_similarities

In [ ]:
all_similarities = torch.zeros(sentence_meanings.shape[0], sentence_meanings.shape[0])
for j in range(sentence_meanings.shape[0]):
  j_similarities = torch.zeros(sentence_meanings.shape[0])

  for i in range(sentence_meanings.shape[0]):
    for k in range(sentence_meanings.shape[1]):
      cs_j_i = sentence_meanings[j][k] * sentence_meanings[i][k]
      j_similarities[i] += cs_j_i

  all_similarities[j] = j_similarities

all_similarities.detach().numpy()

all_similarities [20,20] lik kelimelerin arasındaki benzerliği tutan bir matris

Yukarıdaki uzun hali transpozunu alıp orjinal meannigs ile çarpılınca aynı matris gelir

In [ ]:
sentence_meanings.shape, sentence_meanings.T.shape

In [ ]:
# Query=sentence_meanings, Key=sentence_meanings.T
all_sim_torch = sentence_meanings @ sentence_meanings.T
all_sim_torch

In [ ]:
torch.sum(all_sim_torch[0])

Normalizasyon daha basit işlmler için

In [ ]:
attention_weights = torch.softmax(all_similarities, dim=1)
attention_weights

Nöronlar arasındaki bağı attention_weights olarak düşünebilirsin. Bunu cümlenin anlamı ile çarptığımızda kelime için, cümledeki diğer kelimeler ne kadar önemli?"

In [ ]:
sentence_context_vector = attention_weights @ sentence_meanings # Value
sentence_context_vector, sentence_meanings

In [ ]:
sentence_meanings_without_pos = u_embedding.embedding(tokens)

In [ ]:
u_sentences = [
  {
    "words": sentence_meanings_without_pos.detach().numpy(),
    "labels": u_tokenizer.tokenize(prompt),
    "color": "blue",
  },
  {
    "words": sentence_meanings.detach().numpy(),
    "labels": u_tokenizer.tokenize(prompt),
    "color": "purple",
  },
  {
    "words": sentence_context_vector.detach().numpy(),
    "labels": u_tokenizer.tokenize(prompt),
    "color": "orange",
  },
]

plot_tokens(u_sentences, "Models Attention Sentence Space")